In [3]:
# Cell 1: Autoreload setup — picks up changes to src/ files without kernel restart
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

print("Setup complete")

Setup complete


## Phase 2 — Feature Engineering

### 1. Building Statistics Consolidation

EDA (Phase 1) found ~20 building/apartment statistic columns clustered in 
`_AVG`/`_MODE`/`_MEDI` triplets, co-missing due to a shared root cause 
(absence of the applicant's building record). This step consolidates each 
triplet to a single `_AVG` column and adds one `BUILDING_INFO_AVAILABLE` flag.

In [4]:
# Cell 2: Load fresh data and apply building-stat consolidation
from feature_engineering import consolidate_building_stats

train_raw = pd.read_csv('../data/raw/home-credit-default-risk/application_train.csv')
print(f"Shape before: {train_raw.shape}")

train_fe = consolidate_building_stats(train_raw)
print(f"Shape after: {train_fe.shape}")
print(train_fe['BUILDING_INFO_AVAILABLE'].value_counts())

Shape before: (307511, 122)
Found 14 building-stat triplets: ['APARTMENTS', 'BASEMENTAREA', 'YEARS_BEGINEXPLUATATION', 'YEARS_BUILD', 'COMMONAREA', 'ELEVATORS', 'ENTRANCES', 'FLOORSMAX', 'FLOORSMIN', 'LANDAREA', 'LIVINGAPARTMENTS', 'LIVINGAREA', 'NONLIVINGAPARTMENTS', 'NONLIVINGAREA']
Dropped 28 redundant MODE/MEDI columns
Kept 14 _AVG columns + 1 new BUILDING_INFO_AVAILABLE flag
Shape after: (307511, 95)
BUILDING_INFO_AVAILABLE
1    158701
0    148810
Name: count, dtype: int64


**Correction:** the initial flag used only `APARTMENTS_AVG` as a reference column, 
which undercounted availability — some applicants had data in other triplets (e.g. 
`YEARS_BEGINEXPLUATATION_AVG`) even when missing `APARTMENTS_AVG`. Fixed to check 
**any** of the 14 `_AVG` columns via `.any(axis=1)`, which more accurately captures 
"does this applicant have any building record at all."

**Corrected result:** 158,701 applicants (51.6%) have at least partial building 
info vs. 148,810 (48.4%) with none — a near-even split, revised from the earlier 
(undercounted) 156,061/151,450 split.

In [5]:
# Cell 3: Sanity-check the new flag
print(train_fe['BUILDING_INFO_AVAILABLE'].value_counts())
print(f"\nRetained columns sample:")
print([c for c in train_fe.columns if 'AVG' in c or 'BUILDING' in c])

BUILDING_INFO_AVAILABLE
1    158701
0    148810
Name: count, dtype: int64

Retained columns sample:
['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG', 'BUILDING_INFO_AVAILABLE']


In [6]:
# Cell 4: Does building info availability correlate with default risk?
check = train_fe.groupby('BUILDING_INFO_AVAILABLE')['TARGET'].agg(['count', 'mean'])
check.columns = ['count', 'default_rate']
check['default_rate_pct'] = check['default_rate'] * 100
print(check)

                          count  default_rate  default_rate_pct
BUILDING_INFO_AVAILABLE                                        
0                        148810      0.092171          9.217123
1                        158701      0.070000          6.999956


**Finding:** Default rate confirms the same pattern with corrected counts: 9.22% 
(no building info) vs. 7.00% (building info available), both deviating from the 
8.07% baseline in the same direction as before. The relationship is stable across 
both the original and corrected flag logic — strengthens confidence this is a real 
signal (likely tied to housing stability/homeownership) rather than a counting 
artifact. Flag retained as a feature for Phase 3.

### 2. EXT_SOURCE Missingness Flags

EDA found that missingness in `EXT_SOURCE_1` and `EXT_SOURCE_3` carries predictive 
signal beyond their raw values (8.52% vs 7.50% default rate for EXT_SOURCE_1; 
9.31% vs 7.77% for EXT_SOURCE_3). Creating binary flags here, before any imputation, 
preserves this signal as a standalone feature. `EXT_SOURCE_2` excluded — only 0.21% 
missing, too rare to be a useful flag.

In [7]:
# Cell 5: Apply EXT_SOURCE missingness flags
from feature_engineering import add_ext_source_missing_flags

train_fe = add_ext_source_missing_flags(train_fe)
print(f"Shape after: {train_fe.shape}")    

EXT_SOURCE_1_MISSING: 173378 flagged (56.38%)
EXT_SOURCE_3_MISSING: 60965 flagged (19.83%)
Shape after: (307511, 97)


**Result:** Flags created matching EDA findings exactly — `EXT_SOURCE_1_MISSING` 
56.38% (173,378), `EXT_SOURCE_3_MISSING` 19.83% (60,965). Shape: 95 → 97 columns.